# Stage 3 CA-Only Training — Results Inspector

Reads a single run folder produced by `train_stage3_ca.py` and displays all outputs:
- Cytokine groups (inherited from Stage 2 checkpoint)
- Run log
- Learning curves (train vs val)
- SA vs CA attention entropy (SA should be flat — frozen sanity check)
- CA weight norm trajectory (null hypothesis diagnostic)
- Learnability ranking (recomputed from dynamics)
- Confusion entropy ranking
- Cell-type confidence trajectories (SA and CA separately)

In [ ]:
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
from IPython.display import Image, display, Markdown
from scipy.stats import mannwhitneyu, spearmanr

import anndata as ad
from cytokine_mil.analysis.dynamics import (
    aggregate_to_donor_level,
    rank_cytokines_by_learnability,
    compute_confusion_entropy_summary,
)

In [ ]:
_results_root = Path("/cs/labs/mornitzan/yam.arieli/cytokines/cytokines-mil/scripts/exp_train_stage3_ca/results")
_available = sorted([p.name for p in _results_root.iterdir()]) if _results_root.exists() else []
_available

In [ ]:
# ============================================================
# SET THIS — name of the folder inside results/
# e.g. "stage3_ca_seed42_20260325_120000_pid12345"
# ============================================================
RESULTS_FOLDER = _available[-1] if _available else ""

# ============================================================

def _load_run(folder_name: str) -> Path:
    if not folder_name:
        print("Set RESULTS_FOLDER to one of:", _available)
        raise ValueError("RESULTS_FOLDER is empty")
    p = _results_root / folder_name
    if not p.exists():
        raise FileNotFoundError(f"Not found: {p}")
    return p

run_dir = _load_run(RESULTS_FOLDER)
print(f"Inspecting: {run_dir}")

## 1. Cytokine Groups

In [ ]:
with open(run_dir / "cytokine_groups.json") as f:
    groups = json.load(f)

simple   = groups["simple_cytokines"]
complex_ = groups["complex_cytokines"]
seed     = groups.get("bootstrap_seed", "(not stored)")

# Infer seed from folder name if not in json
import re
_m = re.search(r"stage3_ca_seed(\d+)", RESULTS_FOLDER)
if _m:
    seed = int(_m.group(1))

display(Markdown(f"""
**Bootstrap seed:** `{seed}`

| Group | Cytokines |
|-------|----------|
| SIMPLE | {', '.join(simple)} |
| COMPLEX | {', '.join(complex_)} |
"""))

# Stage 2 provenance
stage2_dir_path = (run_dir / "stage2_dir.txt").read_text().strip()
print(f"Stage 2 source: {stage2_dir_path}")

## 2. Run Log

In [ ]:
print((run_dir / "run_log.txt").read_text())

## 3. Standard Figures (saved by script)

### 3.1 Learning Curves (train vs val)

In [ ]:
lc_path = run_dir / f"learning_curves_stage3_ca_{seed}.png"
if lc_path.exists():
    display(Image(str(lc_path)))
else:
    print(f"Not found: {lc_path.name}")

### 3.2 SA vs CA Attention Entropy

**SA (blue, solid):** frozen — entropy should be constant across epochs (sanity check).  
**CA (orange, dashed):** trainable — decreasing entropy means CA focused on specific cells.

In [ ]:
ent_path = run_dir / f"sa_vs_ca_entropy_stage3_ca_{seed}.png"
if ent_path.exists():
    display(Image(str(ent_path)))
else:
    print(f"Not found: {ent_path.name}")

### 3.3 CA Weight Norm Trajectory

**Null hypothesis:** norm stays near initial value (~0.01 × sqrt(params)) — CA contributes nothing.  
**Alternative:** norm grows — CA learned genuine additional signal beyond SA.

In [ ]:
norm_path = run_dir / f"ca_weight_norm_stage3_ca_{seed}.png"
if norm_path.exists():
    display(Image(str(norm_path)))
else:
    print(f"Not found: {norm_path.name}")

## 4. Load Dynamics

In [ ]:
with open(run_dir / "dynamics_stage3.pkl", "rb") as fh:
    dyn = pickle.load(fh)

train_records = dyn["records"]
val_records   = dyn["val_records"]
logged_epochs = dyn["logged_epochs"]
ca_norm_traj  = dyn["ca_weight_norm_trajectory"]
loss_traj     = dyn["loss_trajectory"]
conf_ent_train = dyn["confusion_entropy_trajectory"]
conf_ent_val   = dyn["val_confusion_entropy_trajectory"]

print(f"Train records : {len(train_records)}")
print(f"Val records   : {len(val_records)}")
print(f"Logged epochs : {len(logged_epochs)}  (first={logged_epochs[0]}, last={logged_epochs[-1]})")
print(f"CA norm traj  : {len(ca_norm_traj)} points  init={ca_norm_traj[0]:.6f}  final={ca_norm_traj[-1]:.6f}")

# Verify SA entropy is flat (frozen sanity check)
if train_records:
    sa_trajs = [r["entropy_trajectory"] for r in train_records if r.get("entropy_trajectory")]
    if sa_trajs:
        sa_var = float(np.var(np.concatenate(sa_trajs)))
        status = "OK" if sa_var < 1e-6 else "WARNING — SA is not frozen!"
        print(f"SA entropy variance: {sa_var:.2e}  [{status}]")

## 5. Loss & CA Norm Trajectories (recomputed)

In [ ]:
epochs_all = list(range(1, len(loss_traj) + 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_all, loss_traj, color="steelblue", lw=1.5)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean cross-entropy loss")
axes[0].set_title("Training loss — Stage 3 CA-only\n(mean over mega-batches per epoch)")
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_all, ca_norm_traj, color="darkviolet", lw=2)
axes[1].axhline(ca_norm_traj[0], color="gray", ls="--", lw=0.8, label=f"Initial: {ca_norm_traj[0]:.4f}")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("L2 norm of CA parameters")
axes[1].set_title(
    f"CA weight norm — seed {seed}\n"
    f"Null: stays near {ca_norm_traj[0]:.4f}.  "
    f"Final: {ca_norm_traj[-1]:.4f}  (Δ={ca_norm_traj[-1]-ca_norm_traj[0]:+.4f})"
)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Learnability Ranking

In [ ]:
donor_traj     = aggregate_to_donor_level(train_records)
val_donor_traj = aggregate_to_donor_level(val_records)

learn_result     = rank_cytokines_by_learnability(donor_traj,     exclude=["PBS"])
val_learn_result = rank_cytokines_by_learnability(val_donor_traj, exclude=["PBS"])

ranking     = learn_result["ranking"]
val_auc_map = {cyt: auc for cyt, auc in val_learn_result["ranking"]}

print(f"Cytokine learnability ranking — Stage 3 CA-only  |  seed={seed}")
print(f"Metric: {learn_result['metric_description']}")
print()
print(f"{'Rank':>4}  {'Cytokine':<16}  {'Train AUC':>9}  {'Val AUC':>8}  Group")
print("-" * 58)
for i, (cyt, auc) in enumerate(ranking, 1):
    grp     = "SIMPLE" if cyt in simple else "COMPLEX"
    val_auc = val_auc_map.get(cyt, float("nan"))
    print(f"  {i:2d}.  {cyt:<16}  {auc:>9.3f}  {val_auc:>8.3f}  {grp}")

simple_aucs  = [auc for cyt, auc in ranking if cyt in simple]
complex_aucs = [auc for cyt, auc in ranking if cyt in complex_]
print()
print(f"SIMPLE  mean={np.mean(simple_aucs):.3f}  median={np.median(simple_aucs):.3f}")
print(f"COMPLEX mean={np.mean(complex_aucs):.3f}  median={np.median(complex_aucs):.3f}")

## 7. Hypothesis Test

In [ ]:
auc_map          = {cyt: auc for cyt, auc in ranking}
simple_auc_vals  = [auc_map[c] for c in simple  if c in auc_map]
complex_auc_vals = [auc_map[c] for c in complex_ if c in auc_map]

if simple_auc_vals and complex_auc_vals:
    u_stat, p_one_sided = mannwhitneyu(simple_auc_vals, complex_auc_vals, alternative="greater")
    n1, n2 = len(simple_auc_vals), len(complex_auc_vals)
    r_rb   = 1 - (2 * u_stat) / (n1 * n2)
    supported = p_one_sided < 0.05
    verdict   = "**SUPPORTED** (p < 0.05)" if supported else "**NOT SUPPORTED** (p ≥ 0.05)"

    train_order = [c for c, _ in ranking]
    val_order   = [c for c, _ in val_learn_result["ranking"]]
    val_rank_by = {c: i for i, c in enumerate(val_order)}
    val_aligned = [val_rank_by.get(c, len(val_order)) for c in train_order]
    rho_gen, pval_gen = spearmanr(range(len(train_order)), val_aligned)

    display(Markdown(f"""
**Hypothesis:** SIMPLE cytokines rank higher in learnability AUC than COMPLEX.  
**Test:** one-sided Mann-Whitney U (train donors only).

| | Value |
|---|---|
| SIMPLE AUCs | {', '.join(f'{x:.3f}' for x in sorted(simple_auc_vals, reverse=True))} |
| COMPLEX AUCs | {', '.join(f'{x:.3f}' for x in sorted(complex_auc_vals, reverse=True))} |
| Mann-Whitney U | {u_stat:.1f} |
| p-value (one-sided) | {p_one_sided:.4f} |
| rank-biserial r | {r_rb:.3f} |
| **Verdict** | {verdict} |
| Train/val rank Spearman ρ | {rho_gen:.3f} (p={pval_gen:.3f}) |
| Stable (ρ > 0.7) | {rho_gen > 0.7} |
"""))
else:
    print("Not enough cytokines in one or both groups to run test.")

## 8. Confusion Entropy Ranking

In [ ]:
conf_result     = compute_confusion_entropy_summary(conf_ent_train, exclude=["PBS"])
val_conf_result = compute_confusion_entropy_summary(conf_ent_val,   exclude=["PBS"])
val_conf_map    = {cyt: auc for cyt, auc in val_conf_result["ranking"]}

print(f"Confusion entropy ranking — Stage 3 CA-only  |  seed={seed}")
print(f"Metric: {conf_result['metric_description']}")
print()
print(f"{'Cytokine':<20}  {'Train AUC(H_c)':>14}  {'Val AUC(H_c)':>12}  Group")
print("-" * 64)
for cyt, auc in conf_result["ranking"]:
    grp     = "SIMPLE" if cyt in simple else ("COMPLEX" if cyt in complex_ else "")
    val_auc = val_conf_map.get(cyt, float("nan"))
    print(f"  {cyt:<20}  {auc:>14.3f}  {val_auc:>12.3f}  {grp}")

## 9. SA vs CA Entropy per Cytokine (recomputed from dynamics)

**SA (blue):** frozen — should be a flat horizontal line.  
**CA (orange):** trainable — key metric. Decreasing = CA focusing on specific cells.

In [ ]:
def _extract_layer_entropy(records, cytokine):
    sa_curves, ca_curves = [], []
    for rec in records:
        if rec["cytokine"] != cytokine:
            continue
        sa = rec.get("entropy_trajectory")
        ca = rec.get("entropy_trajectory_ca")
        if sa and ca:
            sa_curves.append(np.array(sa))
            ca_curves.append(np.array(ca))
    return sa_curves, ca_curves


all_cyts = simple + complex_
group_map = {c: "SIMPLE" for c in simple}
group_map.update({c: "COMPLEX" for c in complex_})

n_cyts = len(all_cyts)
ncols = 5
nrows = (n_cyts + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), sharey=False)
axes = np.array(axes).flatten()

for ax, cyt in zip(axes, all_cyts):
    grp = group_map.get(cyt, "")
    sa_curves, ca_curves = _extract_layer_entropy(train_records, cyt)
    if sa_curves:
        mean_sa = np.mean(sa_curves, axis=0)
        mean_ca = np.mean(ca_curves, axis=0)
        ax.plot(logged_epochs, mean_sa, color="steelblue",  lw=2, label="SA (frozen)")
        ax.plot(logged_epochs, mean_ca, color="darkorange", lw=2, ls="--", label="CA (trainable)")
        sa_range = float(max(mean_sa) - min(mean_sa))
        ax.set_title(f"{cyt} ({grp})\nSA Δ={sa_range:.4f}", fontsize=9)
    else:
        ax.set_title(f"{cyt} ({grp})\n[no data]", fontsize=9)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("H(a) [nats]")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.25)

for ax in axes[n_cyts:]:
    ax.set_visible(False)

fig.suptitle(
    f"Stage 3 CA-only — SA vs CA entropy per cytokine  |  seed={seed}\n"
    "SA Δ shows how flat the frozen SA is (should be ≈ 0).  "
    "CA changes only if it learned signal.",
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## 10. Cell-type Confidence Trajectories

Cell-type labels are **never** used during training — reintroduced post-hoc only.

- **Left panel (SA):** primary responder signal — `C_SA_i(t) = a_SA_i(t) * P(t)(Y_correct)`. SA is frozen, so this reflects the Stage 2 representation directly.
- **Right panel (CA):** cascade responder signal — `C_CA_i(t) = a_CA_i(t) * P(t)(Y_correct)`. Only CA parameters are updated.
- Cell types high in CA but low in SA = secondary cascade responders that CA discovered independently.

In [ ]:
print("Loading cell type labels from pseudo-tube h5ad files (post-hoc only)...")
cell_type_obs = {}
for rec in train_records + val_records:
    path = rec["tube_path"]
    if path not in cell_type_obs:
        _adata = ad.read_h5ad(path, backed="r")
        cell_type_obs[path] = _adata.obs["cell_type"].values.copy()
        _adata.file.close()

_all_cts = sorted({ct for labels in cell_type_obs.values() for ct in labels})
print(f"Cell types found ({len(_all_cts)}): {_all_cts}")

In [ ]:
def _compute_ct_trajectories(records, cell_type_obs, cytokine, conf_key):
    """
    Compute mean C_i(t) per cell type for one cytokine.
    Aggregation: median across pseudo-tubes per donor, then mean across donors.
    """
    cyt_records = [r for r in records if r["cytokine"] == cytokine]
    if not cyt_records:
        return {}
    raw = defaultdict(lambda: defaultdict(list))
    for rec in cyt_records:
        conf = rec.get(conf_key)
        if conf is None:
            continue
        ct_labels = cell_type_obs.get(rec["tube_path"])
        if ct_labels is None:
            continue
        conf_arr = np.array(conf)  # (n_cells, n_logged_epochs)
        for ct in np.unique(ct_labels):
            mask = ct_labels == ct
            mean_traj = conf_arr[mask].mean(axis=0)
            raw[str(ct)][rec["donor"]].append(mean_traj)
    result = {}
    for ct, donors in raw.items():
        donor_trajs = [np.median(np.stack(trajs), axis=0) for trajs in donors.values()]
        result[ct] = np.mean(np.stack(donor_trajs), axis=0)
    return result


def _plot_ct_sa_vs_ca(records, cell_type_obs, cytokine, logged_epochs, group=""):
    sa_curves = _compute_ct_trajectories(records, cell_type_obs, cytokine,
                                         conf_key="confidence_trajectory_sa")
    ca_curves = _compute_ct_trajectories(records, cell_type_obs, cytokine,
                                         conf_key="confidence_trajectory_ca")
    if not sa_curves and not ca_curves:
        print(f"No confidence data for {cytokine}.")
        return

    all_cts   = sorted(set(sa_curves) | set(ca_curves))
    cmap      = plt.cm.tab20.colors
    ct_colors = {ct: cmap[i % len(cmap)] for i, ct in enumerate(all_cts)}

    y_max = max(
        max((t.max() for t in sa_curves.values()), default=0),
        max((t.max() for t in ca_curves.values()), default=0),
    ) * 1.05 or 0.1

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    for ax, curves, layer_label in zip(axes, [sa_curves, ca_curves], ["SA layer (frozen)", "CA layer (trainable)"]):
        for ct in all_cts:
            if ct not in curves:
                continue
            ax.plot(logged_epochs, curves[ct], color=ct_colors[ct], lw=1.5, label=ct)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Mean C_i(t)")
        ax.set_title(layer_label)
        ax.set_ylim(0, y_max)
        ax.legend(fontsize=7, ncol=2, loc="upper left")
        ax.grid(alpha=0.25)

    fig.suptitle(
        f"{cytokine}  [{group}] — SA vs CA cell-type confidence trajectories  |  seed={seed}\n"
        "Metric: mean C_i(t) = a_i(t) * P(t)(Y_correct), grouped by cell type, "
        "aggregated to donor level (median per donor, mean across donors)",
        fontsize=9, y=1.02,
    )
    plt.tight_layout()
    plt.show()
    return sa_curves, ca_curves

### 10.1 SA vs CA Cell-type Confidence per Cytokine

- **SA left panel:** reflects Stage 2 learned representation (frozen).
- **CA right panel:** what the newly trained CA layer discovered independently.
- Cell types **high in CA but flat in SA** = secondary cascade targets that required contextual signal.

In [ ]:
ct_results = {}
for cyt in sorted(simple + complex_):
    grp = "SIMPLE" if cyt in simple else "COMPLEX"
    result = _plot_ct_sa_vs_ca(train_records, cell_type_obs, cyt, logged_epochs, group=grp)
    if result is not None:
        ct_results[cyt] = {"sa": result[0], "ca": result[1]}

### 10.2 Cell-type Activation Heatmaps (SA vs CA)

In [ ]:
cyts_ranked = [c for c, _ in ranking]

def _activation_epoch(ct_curves, threshold=0.05):
    result = {}
    for ct, traj in ct_curves.items():
        idx = np.where(np.asarray(traj) >= threshold)[0]
        result[ct] = int(idx[0]) if len(idx) > 0 else None
    return result


def _plot_activation_heatmap(records, cell_type_obs, cyts_ranked, logged_epochs,
                              simple_cyts, complex_cyts, conf_key, layer_label, threshold=0.05):
    all_ct = set()
    curves_by_cyt = {}
    for cyt in cyts_ranked:
        c = _compute_ct_trajectories(records, cell_type_obs, cyt, conf_key=conf_key)
        curves_by_cyt[cyt] = c
        all_ct.update(c.keys())
    all_ct = sorted(all_ct)

    act = {cyt: _activation_epoch(curves_by_cyt[cyt], threshold) for cyt in cyts_ranked}
    matrix = np.full((len(cyts_ranked), len(all_ct)), np.nan)
    for i, cyt in enumerate(cyts_ranked):
        for j, ct in enumerate(all_ct):
            ep_idx = act[cyt].get(ct)
            if ep_idx is not None:
                matrix[i, j] = logged_epochs[ep_idx]

    fig, ax = plt.subplots(figsize=(14, 6))
    vmin = np.nanmin(matrix) if not np.all(np.isnan(matrix)) else 0
    vmax = np.nanmax(matrix) if not np.all(np.isnan(matrix)) else 1
    im = ax.imshow(matrix, aspect="auto", cmap="YlOrRd_r",
                   vmin=vmin, vmax=vmax, interpolation="nearest")
    plt.colorbar(im, ax=ax, label="First epoch crossing threshold (earlier = darker)")
    ax.set_xticks(range(len(all_ct)))
    ax.set_xticklabels(all_ct, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(cyts_ranked)))
    ylabels = []
    for cyt in cyts_ranked:
        if cyt in simple_cyts:    ylabels.append(f"{cyt}  [SIMPLE]")
        elif cyt in complex_cyts: ylabels.append(f"{cyt}  [COMPLEX]")
        else:                      ylabels.append(cyt)
    ax.set_yticklabels(ylabels, fontsize=8)
    ax.set_xlabel("Cell type")
    ax.set_title(
        f"{layer_label} — cell-type activation epoch (threshold={threshold})  "
        "Rows ordered by learnability AUC (high to low)\n"
        "Metric: first epoch where mean C_i(t) ≥ threshold, aggregated to donor level",
        fontsize=8,
    )
    plt.tight_layout()
    plt.show()
    return act


act_sa = _plot_activation_heatmap(
    train_records, cell_type_obs, cyts_ranked, logged_epochs,
    simple, complex_, conf_key="confidence_trajectory_sa",
    layer_label="SA layer (frozen — Stage 2 representation)",
)
act_ca = _plot_activation_heatmap(
    train_records, cell_type_obs, cyts_ranked, logged_epochs,
    simple, complex_, conf_key="confidence_trajectory_ca",
    layer_label="CA layer (trainable — Stage 3 CA-only)",
)

### 10.3 Cascade Ordering per Cytokine (SA vs CA)

In [ ]:
print(f"Cell-type cascade ordering — SA vs CA  |  seed={seed}")
print("Metric: first epoch index where mean C_i(t) >= 0.05 per layer")
print("SA-only = direct targets; CA-only = secondary cascade; both = hub")
print()

for cyt in cyts_ranked:
    grp = "SIMPLE" if cyt in simple else "COMPLEX"
    sa_act = act_sa.get(cyt, {})
    ca_act = act_ca.get(cyt, {})

    sa_activated = {ct for ct, ep in sa_act.items() if ep is not None}
    ca_activated = {ct for ct, ep in ca_act.items() if ep is not None}

    sa_only = sorted(sa_activated - ca_activated)
    ca_only = sorted(ca_activated - sa_activated)
    both    = sorted(sa_activated & ca_activated)

    print(f"{cyt:<20} [{grp}]")
    if sa_only: print(f"  SA-only (direct targets):       {sa_only}")
    if ca_only: print(f"  CA-only (cascade responders):   {ca_only}")
    if both:    print(f"  Both (hub cells):               {both}")
    if not sa_only and not ca_only and not both:
        print("  (no cell types crossed threshold)")
    print()